# 01 — MediaPipe Landmark Extraction

Extracts hand + pose landmarks from raw video files.

**Input:** `Drive/SignLanguage_Project/00_Raw_Videos/`  
**Output:** `Drive/SignLanguage_Project/01_Landmarks/`

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Install dependencies ───────────────────────
!pip install mediapipe opencv-python-headless tqdm -q

In [ ]:
# ── 3. Clone repo & import modules ────────────────
import sys
sys.path.append('/content/sign-language-pipeline')

from src.mediapipe_extractor import process_video
from src.pipeline_utils import get_video_paths, log_processing_result
import numpy as np, os
from tqdm import tqdm

In [ ]:
# ── 4. Configure paths ────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/SignLanguage_Project'
RAW_DIR    = f'{DRIVE_BASE}/00_Raw_Videos'
LAND_DIR   = f'{DRIVE_BASE}/01_Landmarks'
LOG_PATH   = f'{DRIVE_BASE}/03_Docs/data_log.csv'

os.makedirs(LAND_DIR, exist_ok=True)

In [ ]:
# ── 5. Extract landmarks from all videos ──────────
video_paths = get_video_paths(RAW_DIR)
print(f'Found {len(video_paths)} videos')

for video_path, label in tqdm(video_paths):
    sequence = process_video(video_path, target_frames=30)
    
    if sequence is None:
        log_processing_result(LOG_PATH, video_path, label, 'FAILED', 'extraction error')
        continue
    
    # Save .npy per video
    out_dir = os.path.join(LAND_DIR, label)
    os.makedirs(out_dir, exist_ok=True)
    fname = os.path.splitext(os.path.basename(video_path))[0] + '.npy'
    np.save(os.path.join(out_dir, fname), sequence)
    log_processing_result(LOG_PATH, video_path, label, 'OK')

print('Extraction complete!')